# GLOBAL IMPORTS AND CONFIGS

INGESTION PIPELINE

In [1]:
from __future__ import annotations

from pathlib import Path
import os
import sys
import time

import pandas as pd

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Ensure Chroma persists to data/chroma_db
PERSIST_DIR = PROJECT_ROOT / "data" / "chroma_db"
os.environ["CHROMA_PERSIST_DIR"] = str(PERSIST_DIR)

from app.core.config import get_settings
from app.core.logging import configure_logging, get_logger
from app.services.embedding_service import get_embedding_service
from app.services.vector_store_service import VectorStoreService
from app.utils.chunker import JobDescriptionChunker
from app.utils.preprocessor import build_chunk_document
from tqdm  import tqdm

settings = get_settings()
configure_logging(settings.log_level)
logger = get_logger("notebook.ingestion")

DATA_PATH = PROJECT_ROOT / "data" / "LFJobs.csv"
BATCH_SIZE = 128
DRY_RUN = False
LIMIT = None

C:\Users\mahar\miniconda3\envs\LeapFrog_Ass2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 1) Load and validate CSV

In [2]:
REQUIRED_COLUMNS = {
    "ID",
    "Job Category",
    "Job Title",
    "Company Name",
    "Publication Date",
    "Job Location",
    "Job Level",
    "Job Description",
}

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Missing data file: {DATA_PATH}")

df = pd.read_csv(DATA_PATH, dtype=str).fillna("")
missing = REQUIRED_COLUMNS - set(df.columns)
if missing:
    raise ValueError(f"CSV missing required columns: {sorted(missing)}")

if LIMIT:
    df = df.head(LIMIT)
    logger.info("limit_applied", limit=LIMIT)

before_rows = len(df)
# Enforce required fields (ID and Job Description)
mask_required = df["ID"].str.strip().ne("") & df["Job Description"].str.strip().ne("")
df = df.loc[mask_required].copy()

dropped = before_rows - len(df)
if dropped:
    logger.info("rows_dropped_missing_required_fields", dropped=dropped)

logger.info("csv_loaded", rows=len(df), columns=list(df.columns))

{"rows": 1000, "columns": ["ID", "Job Category", "Job Title", "Company Name", "Publication Date", "Job Location", "Job Level", "Tags", "Job Description"], "event": "csv_loaded", "level": "info", "timestamp": "2026-05-19T13:29:41.654738Z"}


# 2) Preprocess + build documents

In [3]:

def build_documents(frame: pd.DataFrame) -> list[dict]:
    documents: list[dict] = []
    for _, row in frame.iterrows():
        row_dict = row.to_dict()
        text = build_chunk_document(row_dict)
        metadata = {
            "job_id": str(row_dict.get("ID", "")),
            "job_title": str(row_dict.get("Job Title", "")),
            "company_name": str(row_dict.get("Company Name", "")),
            "job_category": str(row_dict.get("Job Category", "")),
            "job_level": str(row_dict.get("Job Level", "")),
            "job_location": str(row_dict.get("Job Location", "")),
            "publication_date": str(row_dict.get("Publication Date", "")),
            "tags": str(row_dict.get("Tags", "")),
        }
        documents.append({"text": text, "metadata": metadata})
    return documents


documents = build_documents(df)
logger.info("documents_built", count=len(documents))

{"count": 1000, "event": "documents_built", "level": "info", "timestamp": "2026-05-19T13:29:44.549204Z"}


# 3) Chunk (stable IDs = hash of job_id + chunk_index)

In [4]:
chunker = JobDescriptionChunker()
all_chunks = chunker.chunk_batch(documents)

logger.info(
    "chunking_complete",
    num_jobs=len(documents),
    total_chunks=len(all_chunks),
    avg_chunks_per_job=round(len(all_chunks) / max(len(documents), 1), 1),
)

if all_chunks:
    print("Sample chunk ID:", all_chunks[0]["id"])

{"chunk_size": 512, "chunk_overlap": 64, "event": "chunker_initialised", "level": "info", "timestamp": "2026-05-19T13:29:44.599302Z"}
{"num_documents": 1000, "total_chunks": 16501, "event": "batch_chunked", "level": "info", "timestamp": "2026-05-19T13:29:44.729723Z"}
{"num_jobs": 1000, "total_chunks": 16501, "avg_chunks_per_job": 16.5, "event": "chunking_complete", "level": "info", "timestamp": "2026-05-19T13:29:44.731243Z"}
Sample chunk ID: 12d3be1ed941447da84180c03d8f48e0


# 4) Embed + store in ChromaDB

In [5]:
if DRY_RUN:
    print("DRY_RUN=True: skipping embedding and storage")
    if all_chunks:
        sample = all_chunks[0]
        print("\n=== Sample chunk ===")
        print("ID:", sample["id"])
        print("Metadata:", sample["metadata"])
        print("Text preview:\n", sample["text"][:400])
else:
    t_start = time.perf_counter()
    embedding_service = get_embedding_service()
    vector_store = VectorStoreService(embedding_service=embedding_service)

    before_count = vector_store.document_count
    vector_store.add_chunks(all_chunks, batch_size=BATCH_SIZE)
    after_count = vector_store.document_count

    elapsed = time.perf_counter() - t_start
    logger.info(
        "ingestion_complete",
        chunks_before=before_count,
        chunks_after=after_count,
        new_chunks=after_count - before_count,
        elapsed_seconds=round(elapsed, 1),
    )
    print(
        f"\nIngestion complete in {elapsed:.1f}s\n"
        f"Jobs processed : {len(documents)}\n"
        f"Total chunks   : {len(all_chunks)}\n"
        f"Store size     : {after_count} chunks\n"
    )

{"model": "BAAI/bge-small-en-v1.5", "device": "cpu", "event": "loading_embedding_model", "level": "info", "timestamp": "2026-05-19T13:29:44.757517Z"}
{"model": "BAAI/bge-small-en-v1.5", "dimension": 384, "event": "embedding_model_ready", "level": "info", "timestamp": "2026-05-19T13:29:48.566601Z"}
{"persist_dir": "C:\\Users\\mahar\\PycharmProjects\\PythonProject1\\data\\chroma_db", "event": "initialising_chroma", "level": "info", "timestamp": "2026-05-19T13:29:48.568283Z"}
{"collection": "job_listings", "num_documents": 16501, "event": "chroma_ready", "level": "info", "timestamp": "2026-05-19T13:29:48.731393Z"}
{"total": 16501, "batch_size": 128, "event": "adding_chunks", "level": "info", "timestamp": "2026-05-19T13:29:48.736384Z"}
{"num_texts": 128, "batch_size": 128, "event": "embedding_documents", "level": "info", "timestamp": "2026-05-19T13:29:48.737765Z"}
{"num_texts": 128, "batch_size": 128, "event": "embedding_documents", "level": "info", "timestamp": "2026-05-19T13:29:52.756097